In [1]:
import torch
import numpy as np
from dust3r.model_w3dgs import AsymmetricCroCo3DStereo3DGS
import torchvision.transforms as tvf

Warning, cannot find cuda-compiled version of RoPE2D, using a slow pytorch version instead
Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


Using cache found in /nethome/abati7/.cache/torch/hub/intel-isl_MiDaS_master
/nethome/abati7/flash/miniconda3/envs/duster/lib/python3.11/site-packages/timm/models/_factory.py:117: UserWarning: Mapping deprecated model name vit_base_resnet50_384 to current vit_base_r50_s16_384.orig_in21k_ft_in1k.
  model = create_fn(
Using cache found in /nethome/abati7/.cache/torch/hub/intel-isl_MiDaS_master


In [2]:
ImgNorm = tvf.Compose([tvf.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
inf = float('inf')
model = AsymmetricCroCo3DStereo3DGS(pos_embed='RoPE100', 
                                    patch_embed_cls='ManyAR_PatchEmbed', 
                                    img_size=(224, 224), 
                                    head_type='dpt_3dgs', 
                                    output_mode='pts3d', 
                                    depth_mode=('exp', -inf, inf), 
                                    conf_mode=('exp', 1, inf), 
                                    enc_embed_dim=1024, 
                                    enc_depth=24, 
                                    enc_num_heads=16, 
                                    dec_embed_dim=768, 
                                    dec_depth=12, 
                                    dec_num_heads=12)

RuntimeError: Found no NVIDIA driver on your system. Please check that you have an NVIDIA GPU and installed a driver from http://www.nvidia.com/Download/index.aspx

In [3]:
img1 = torch.rand((3,224,224))
img2 = torch.rand((3,224,224))

In [4]:
imgs = []
imgs.append((dict(img=ImgNorm(img1)[None], true_shape=np.int32(
            [[224,224]]), idx=0, instance='0'),
            
            dict(img=ImgNorm(img2)[None], true_shape=np.int32(
            [[224,224]]), idx=1, instance='1')))

In [5]:
from dust3r.utils.device import to_cpu, collate_with_cat

In [6]:
view1, view2 = collate_with_cat(imgs)

In [9]:
print(view1['img'].shape)

torch.Size([1, 3, 224, 224])


In [7]:
pred1, pred2 = model(view1, view2)

In [8]:
def getAttShapes(pred):
    a = {}
    for k in pred:
        a[k]=(pred1[k].shape)
    return a

In [9]:
getAttShapes(pred1)

{'pts3d': torch.Size([1, 224, 224, 3]),
 'conf': torch.Size([1, 224, 224]),
 'alpha': torch.Size([1, 224, 224]),
 'quad': torch.Size([1, 224, 224, 4]),
 'scale': torch.Size([1, 224, 224, 3]),
 'shsRGB': torch.Size([1, 224, 224, 48])}

In [1]:
import sys
sys.path.append("/nethome/abati7/flash/Work/recon/dust3r/dependencies/FSGS/")

In [2]:
from gaussian_renderer import render

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


Using cache found in /nethome/abati7/.cache/torch/hub/intel-isl_MiDaS_master
/nethome/abati7/flash/miniconda3/envs/duster/lib/python3.11/site-packages/timm/models/_factory.py:117: UserWarning: Mapping deprecated model name vit_base_resnet50_384 to current vit_base_r50_s16_384.orig_in21k_ft_in1k.
  model = create_fn(
Using cache found in /nethome/abati7/.cache/torch/hub/intel-isl_MiDaS_master


In [12]:
from scene.cameras import Camera
from scene.gaussian_model import GaussianModel
from utils.general_utils import inverse_sigmoid
from arguments import PipelineParams

In [13]:
w2c = np.random.rand(4,4)
R = np.transpose(w2c[:3,:3])  # R is stored transposed due to 'glm' in CUDA code
T = w2c[:3, 3]

In [ ]:
viewpoint_cam = Camera(colmap_id=0, R=R, T=T, 
                FoVx=np.pi/3, FoVy=np.pi/3,  image=img1, gt_alpha_mask=None,
                uid=0, data_device="cuda:0", image_name='testing',
                depth_image=None, mask=None, bounds=None) #most of these are not needed